## Initialization

In [1]:
#initializing workspace

import pandas as pd
import matplotlib.pyplot as plt
import trxtools as tt
from trxtools.utils import files
from trxtools.utils import names
import seaborn as sns
import trxtools.profiles.metaprofiles as meta
import os
from pyCRAC.Parsers import GTF2
import pyBigWig
import numpy as np
from trxtools.profiles.BigWig import *
from scipy.signal import argrelextrema

pathGTF = "../references/hg41.ncbiRefSeq.rRNA.gtf"
gtf = GTF2.Parse_GTF()
gtf.read_GTF(pathGTF)

## optional; human fasta file not provided due to large size
# pathTAB = "../references/hg41.ncbiRefSeq.rRNA.tab"
# rDNA_fasta = gtf.readTAB(pathTAB)

bed = files.gtf2bed(pathGTF)

ModuleNotFoundError: No module named 'pyCRAC'

# rRNA Profile analysis

This workflow demonstrates how to visualize read coverage for a single gene across multiple experiments, using the example of 45S pre-rRNA. The approach combines BigWig-based coverage extraction with preprocessing and normalization, followed by summarization as box plots. This method is particularly useful for comparing variability and consistency between replicates or experimental conditions. By focusing on a single gene, you can assess technical reproducibility and detect systematic differences in processing or treatment effects.

## Steps to Plot Coverage for a Single Gene (e.g., 45S pre-rRNA) Across Multiple Experiments
### List BigWig Files
Collect all BigWig files corresponding to your experiments or replicates. Organize them in a directory and create a mapping of experiment names to file paths.

### Extract Gene Coverage with geneFromBigWig
Use the geneFromBigWig function to load coverage profiles for the target gene (e.g., 45S pre-rRNA) from all BigWig files. This produces a matrix where rows represent positions within the gene and columns represent experiments.

### Preprocess Coverage Data
Apply preprocessing using tt.profiles.preprocess or equivalent functions. Common options include:

Raw: Keep original coverage values.
Normalization: Scale by total reads (e.g., CPM) for comparability across experiments.
Pseudocounts: Add a small constant to avoid zeros, especially before log transformation.

### Summarize and Visualize with plot_as_box_plot
Pass the processed data to plot_as_box_plot to generate box plots. Each box represents the distribution of coverage values for the gene in one experiment, allowing easy comparison of variability and central tendencies.

In [ ]:
# Listing rRNA genes
l01_45S = ["RNA45SN1", "RNA45SN2", "RNA45SN3"]
l02_5ETS = ["RNA18SN1", "RNA18SN2", "RNA18SN3"]
l03_18S = ["RNA18SN1", "RNA18SN2", "RNA18SN3"]
l04_25S = ["RNA25SN1", "RNA25SN2", "RNA25SN3"]
l05_5S = ["RNA5S"+str(i) for i in range(1, 17+1)]  # RNA5S1 to RNA5S17
# l06_5Spseudogenes = ["RNA5SP"+str(i) for i in range(18, 536+1)]  # RNA5SP18 to RNA5SP536 - some pseudogenes are missing

# Processing sites for 45S pre-rRNA
df00_RNAcoordinates = pd.DataFrame()
for i in ['N1', 'N2', 'N3']:
    start = gtf.chromosomeCoordinates(f'RNA45S{i}')[0]
    start18 = gtf.chromosomeCoordinates(f'RNA18S{i}')[0]
    end18 = gtf.chromosomeCoordinates(f'RNA18S{i}')[1]
    start58 = gtf.chromosomeCoordinates(f'RNA5-8S{i}')[0]
    end58 = gtf.chromosomeCoordinates(f'RNA5-8S{i}')[1]
    start28 = gtf.chromosomeCoordinates(f'RNA28S{i}')[0]
    end28 = gtf.chromosomeCoordinates(f'RNA28S{i}')[1]
    end = gtf.chromosomeCoordinates(f'RNA45S{i}')[1]
    df00_RNAcoordinates[f'RNA45S{i}'] = [start,
                                        start18,
                                        end18,
                                        start58,
                                        end58,
                                        start28,
                                        end28,
                                         end]

df00_RNAcoordinates.index = ['start_RNA45', 'start_RNA18', 'end_RNA18', 'start_RNA5-8', 'end_RNA5-8', 'start_RNA28', 'end_RNA28', 'end_RNA45']
df00_RNAcoordinates = df00_RNAcoordinates.T
df00_RNAcoordinates = df00_RNAcoordinates.subtract(df00_RNAcoordinates['start_RNA45'], axis=0)
df00_RNAcoordinates

# Cyclon profile along RNA45S N1

In [ ]:
bwDir_04 = "../04_BigWig/"

#listing datasets
data_files_read_04 = [f for f in os.listdir(bwDir_04) if 'read' in f and "bw" in f]

# print(len(os.listdir(bwDir_04)))
# print(len(data_files_read_04))

flank = 0

df01_RNA45S_read_04 = geneFromBigWig(gene_name="RNA45SN1", data_path=bwDir_04,
                                     data_files=data_files_read_04, gtf=gtf,ranges=flank)

df11_RNA45SN1 = tt.profiles.profileTools.preprocess(df01_RNA45S_read_04,
                                     let_in=['Cyclon'], 
                                     let_out=["CPM", "Cyclon-",'del0'],stats=True)

tt.plotting.plot_as_box_plot(df11_RNA45SN1, figsize=(10,3), title="Cyclon reads along rDNA (04_BigWig)",
                            #  start=12000,stop=13000,
                             dpi=300,ylim=(None,0.003),offset=0,fname='Fig_01_RNA45SN1.png')

### 45S pre-rRNA for all three genes with processing sites marked

In [ ]:
for i in ['N1', 'N2', 'N3']:
    df_temp_datasets = geneFromBigWig(gene_name=f"RNA45S{i}", data_path=bwDir_04, data_files=data_files_read_04, gtf=gtf,ranges=flank)
    df_temp_profile = tt.profiles.profileTools.preprocess(df_temp_datasets,
                                let_in=['Cyclon_none'], let_out=["CPM",'del0'],stats=True)
    processing_sites = df00_RNAcoordinates.loc[f'RNA45S{i}'].to_list()
    tt.plotting.plot_as_box_plot(df_temp_profile, figsize=(10,3), title=f"Cyclon reads along rDNA (RNA45S{i})",
                                h_lines=processing_sites,
                             dpi=300,ylim=(None,0.003),offset=100,fname=f'Fig_02_RNA45S{i}.png')

## deletions

In [ ]:
#listing datasets
data_files_del_04 = [f for f in os.listdir(bwDir_04) if 'del0' in f and "bw" in f]

# for i in ['N1', 'N2', 'N3']:
for i in ['N1']:
    df_temp_datasets = geneFromBigWig(gene_name=f"RNA45S{i}", data_path=bwDir_04, data_files=data_files_del_04, gtf=gtf,ranges=flank)
    df_temp_profile = tt.profiles.profileTools.preprocess(df_temp_datasets,
                                let_in=['Cyclon_none'], let_out=["CPM"],stats=True)
    processing_sites = df00_RNAcoordinates.loc[f'RNA45S{i}'].to_list()
    tt.plotting.plot_as_box_plot(df_temp_profile, figsize=(10,3), title=f"Cyclon deletions along rDNA (RNA45S{i})",
                                h_lines=processing_sites, color='darkred',
                             dpi=300,ylim=(None,0.05),offset=100,fname=f'Fig_02_RNA45S{i}_deletions.png')

### 5' ETS of pre-45S rRNA

In [ ]:
# for i in ['N1', 'N2', 'N3']:
for i in ['N1']:
    df_temp_datasets = geneFromBigWig(gene_name=f"RNA45S{i}", data_path=bwDir_04, data_files=data_files_read_04, gtf=gtf,ranges=flank)
    df_temp_profile = tt.profiles.profileTools.preprocess(df_temp_datasets,
                                     let_in=['Cyclon_none'], let_out=['CPM','del0'],stats=True)
    processing_sites = df00_RNAcoordinates.loc[f'RNA45S{i}'].to_list()
    tt.plotting.plot_as_box_plot(df_temp_profile, figsize=(10,3), title=f"Cyclon reads along 5' ETS (RNA45S{i})",
                                 stop=4000,h_lines=processing_sites,
                             dpi=300,ylim=(None,0.0003),offset=100,fname=f'Fig_02_5ETS{i}.png')

# 5S rRNA

In [ ]:
# for i in ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16','17']:
for i in ['9']:
    df_temp_datasets = geneFromBigWig(gene_name=f"RNA5S{i}", data_path=bwDir_04, data_files=data_files_read_04, gtf=gtf,ranges=flank)
    df_temp_profile = tt.profiles.profileTools.preprocess(df_temp_datasets,
                                     let_in=['Cyclon_none'], let_out=['CPM','del0'],stats=True)
    
    tt.plotting.plot_as_box_plot(df_temp_profile, figsize=(10,3), title=f"Cyclon reads along 5S RNA (RNA5S{i})",
                             dpi=300,ylim=(None,0.03),offset=100,fname=f'Fig_02_5S{i}.png')

## deletions

In [ ]:
#listing datasets
data_files_del_04 = [f for f in os.listdir(bwDir_04) if 'del0' in f and "bw" in f]

flank = 0
df_temp_datasets = geneFromBigWig(gene_name=f"RNA5S9",
                                  data_path=bwDir_04, data_files=data_files_del_04, 
                                  gtf=gtf,ranges=flank)
df_temp_profile = tt.profiles.profileTools.preprocess(df_temp_datasets,
                                     let_in=['Cyclon_none'], 
                                     let_out=['CPM'],stats=True, window=5)

tt.plotting.plot_as_box_plot(df_temp_profile, figsize=(10,3), title=f"Cyclon reads along 5S RNA (RNA5S9)",
                             dpi=300,ylim=(None,0.25),offset=0,fname=f'Fig_02_5SRNA-9_deletions_window5.png', color='darkred')

In [ ]:
#saving raw data as excel file

df_temp_profile = tt.profiles.profileTools.preprocess(df_temp_datasets,
                                     let_in=['Cyclon_none'], pseudocounts_param=False,
                                     ntotal_param=False,
                                     let_out=['CPM'],stats=False, window=1)
df_temp_profile.to_excel('Fig_02_5SRNA-9_deletions_window5.xlsx')

## Comparing truncated Cyclon proteins with wt

In [ ]:
# for i in ['N1', 'N2', 'N3']:
for i in ['N1']:
    df_temp_datasets = geneFromBigWig(gene_name=f"RNA45S{i}", data_path=bwDir_04, data_files=data_files_read_04, gtf=gtf,ranges=flank)
    
    df_temp_profile = tt.profiles.profileTools.preprocess(df_temp_datasets,
                                let_in=['Cyclon_none'], let_out=["CPM",'del0'],stats=True)
    df_temp_profile_CT = tt.profiles.profileTools.preprocess(df_temp_datasets,
                                let_in=['Cyclon-CT'], let_out=["CPM",'del0'],stats=True)
    df_temp_profile_NT = tt.profiles.profileTools.preprocess(df_temp_datasets,
                                let_in=['Cyclon-NT'], let_out=["CPM",'del0'],stats=True)
    processing_sites = df00_RNAcoordinates.loc[f'RNA45S{i}'].to_list()

    tt.plotting.plot_to_compare(df_temp_profile, df_temp_profile_NT, figsize=(10,3), title=f"Cyclon-NT reads along rDNA (RNA45S{i})",
                                # h_lines=processing_sites,
                             dpi=300,ylim=(None,0.003),offset=100,fname=f'Fig_02_RNA45S{i}_compare_NT.png')
    tt.plotting.plot_to_compare(df_temp_profile, df_temp_profile_CT, figsize=(10,3), title=f"Cyclon-CT reads along rDNA (RNA45S{i})",
                                # h_lines=processing_sites,
                             dpi=300,ylim=(None,0.003),offset=100,fname=f'Fig_02_RNA45S{i}_compare_CT.png')

In [ ]:
# for i in ['N1', 'N2', 'N3']:
for i in ['N1']:
    df_temp_datasets = geneFromBigWig(gene_name=f"RNA45S{i}", data_path=bwDir_04, data_files=data_files_read_04, gtf=gtf,ranges=flank)
    
    df_temp_profile = tt.profiles.profileTools.preprocess(df_temp_datasets,
                                let_in=['Cyclon_none'], let_out=["CPM",'del0'],stats=True)
    df_temp_profile_CT = tt.profiles.profileTools.preprocess(df_temp_datasets,
                                let_in=['Cyclon-CT'], let_out=["CPM",'del0'],stats=True)
    df_temp_profile_NT = tt.profiles.profileTools.preprocess(df_temp_datasets,
                                let_in=['Cyclon-NT'], let_out=["CPM",'del0'],stats=True)
    processing_sites = df00_RNAcoordinates.loc[f'RNA45S{i}'].to_list()

    tt.plotting.plot_to_compare(df_temp_profile, df_temp_profile_NT, figsize=(10,3), title=f"Cyclon-NT reads along the 5' ETS rDNA (RNA45S{i})",
                                # h_lines=processing_sites,
                             dpi=300,ylim=(None,0.0003),offset=100,fname=f'Fig_02_RNA45S{i}_compare_NT.png',
                             stop=1000,)
    tt.plotting.plot_to_compare(df_temp_profile, df_temp_profile_CT, figsize=(10,3), title=f"Cyclon-CT reads along the 5' ETS rDNA (RNA45S{i})",
                                # h_lines=processing_sites,
                             dpi=300,ylim=(None,0.0003),offset=100,fname=f'Fig_02_RNA45S{i}_compare_CT.png',
                             stop=1000 )

## 5S rRNA

In [ ]:
# for i in ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16','17']:
for i in ['9']:
    df_temp_datasets = geneFromBigWig(gene_name=f"RNA5S{i}", data_path=bwDir_04, data_files=data_files_read_04, gtf=gtf,ranges=flank)
    
    df_temp_profile = tt.profiles.profileTools.preprocess(df_temp_datasets,
                                let_in=['Cyclon_none'], let_out=["CPM",'del0'],stats=True)
    df_temp_profile_CT = tt.profiles.profileTools.preprocess(df_temp_datasets,
                                let_in=['Cyclon-CT'], let_out=["CPM",'del0'],stats=True)
    df_temp_profile_NT = tt.profiles.profileTools.preprocess(df_temp_datasets,
                                let_in=['Cyclon-NT'], let_out=["CPM",'del0'],stats=True)

    tt.plotting.plot_to_compare(df_temp_profile, df_temp_profile_NT, figsize=(10,3), title=f"Cyclon-NT reads along the 5' ETS rDNA (RNA5S{i})",
                             dpi=300,ylim=(None,0.03),offset=100,fname=f'Fig_02_RNA5S{i}_compare_NT.png')
    tt.plotting.plot_to_compare(df_temp_profile, df_temp_profile_CT, figsize=(10,3), title=f"Cyclon-CT reads along the 5' ETS rDNA (RNA5S{i})",
                             dpi=300,ylim=(None,0.03),offset=100,fname=f'Fig_02_RNA5S{i}_compare_CT.png')

## 5S rRNA metaprofiles and heatmap

In [ ]:
bigwig_plus = [bwDir_04 + file for file in os.listdir(bwDir_04) if file.endswith("_fwd.bw") and "Cyclon_none" in file and "CPM" not in file  and 'del0' not in file]
bigwig_minus = [bwDir_04 + file for file in os.listdir(bwDir_04) if file.endswith("_rev.bw") and "Cyclon_none" in file and "CPM" not in file and 'del0' not in file]

# bed_5S = bed[(bed['gene_name'].str.contains('RNA5S')) & (bed['gene_type'] == 'rRNA')]

mm01 = meta.getMultipleMatrices(
    bigwig_plus, # list of BigWig files for plus strand
    bigwig_minus, # same for minus strand
    bed[bed['gene_type'] == 'rRNA'].drop(columns='gene_type'), # BED with features of interes
    flank_5=100, # We can include flanks around our genes
    flank_3=100
)
mp01 = meta.metaprofile(
    mm01,
    agg_type='mean', # for each position, calculate mean across all genes
    normalize_internal=False # whether to normalize each position to the sum of all positions
)

mp01 = names.cleanNames(mp01, ['../04_BigWig/AG241010_', '_none_120mJ_', '_read_plus.bw'])
mp01['mean'] = mp01.mean(axis=1)
fig, ax = plt.subplots(figsize=(5, 3), layout='constrained')
#plot each library
for col in mp01.columns[:-1]:
    sns.lineplot(data=mp01, x=mp01.index, y=col, label=col, ax=ax, c='grey', alpha=0.6)
# plot mean
sns.lineplot(data=mp01, x=mp01.index, y=mp01.columns[-1], label=mp01.columns[-1], ax=ax, c='orangered')
plt.ylabel('Normalized coverage')
plt.xlabel('Position [nt]')
plt.title("5S rRNA genes")
# plt.ylim(0,0.0000012)
plt.xlim(-50, 250)
# plt.legend().set_visible(False)

# Quantitative comparision of 47S rDNA profiles 
### Cyclon vs RPP25L

In [ ]:
bwDir_RPP25L = "../04a_BigWig_umitools/"
#listing datasets
data_files_CPM_Cyclon = [f for f in os.listdir(bwDir_04) if 'CPM' in f and "bw" in f]
data_files_CPM_RPP25L = [f for f in os.listdir(bwDir_RPP25L) if 'CPM' in f and "bw" in f]

flank = 0

df21_Cyclon_CPM = geneFromBigWig(gene_name="RNA45SN1", data_path=bwDir_04,
                                     data_files=data_files_CPM_Cyclon, gtf=gtf,ranges=flank)

df22_RPP25L_CPM = geneFromBigWig(gene_name="RNA45SN1", data_path=bwDir_RPP25L,
                                     data_files=data_files_CPM_RPP25L, gtf=gtf,ranges=flank)

df23_Cyclon_CPM_profile = tt.profiles.profileTools.preprocess(df21_Cyclon_CPM,
                                     let_in=['Cyclon_none','CPM'], 
                                     stats=True, ntotal_param=False)
df24_RPP25L_CPM_profile = tt.profiles.profileTools.preprocess(df22_RPP25L_CPM,
                                     let_in=['RPP25L_none','CPM'], 
                                     stats=True, ntotal_param=False)

tt.plotting.plot_to_compare(df23_Cyclon_CPM_profile, df24_RPP25L_CPM_profile, figsize=(10,3),
                            title=f"Cyclon vs RPP25L reads along rDNA (RNA45SN1)",
                            ref_label='Cyclon',label='RPP25L',
                            dpi=300,ylim=(None,12000),offset=0,fname=f'Fig_03_RNA45SN1_Cyclon_vs_RPP25L.png')
# tt.plotting.plot_as_box_plot(df11_RNA45SN1, figsize=(10,3), title="Cyclon reads along rDNA (04_BigWig)",
#                             #  start=12000,stop=13000,
#                              dpi=300,ylim=(None,0.003),offset=0,fname='Fig_01_RNA45SN1.png')

In [ ]:
flank = 100

df31_Cyclon_CPM = geneFromBigWig(gene_name="RNA5S9", data_path=bwDir_04,
                                     data_files=data_files_CPM_Cyclon, gtf=gtf,ranges=flank)

df32_RPP25L_CPM = geneFromBigWig(gene_name="RNA5S9", data_path=bwDir_RPP25L,
                                     data_files=data_files_CPM_RPP25L, gtf=gtf,ranges=flank)

df33_Cyclon_CPM_profile = tt.profiles.profileTools.preprocess(df31_Cyclon_CPM,
                                     let_in=['Cyclon_none','CPM'], 
                                     stats=True, ntotal_param=False)
df34_RPP25L_CPM_profile = tt.profiles.profileTools.preprocess(df32_RPP25L_CPM,
                                     let_in=['RPP25L_none','CPM'], 
                                     stats=True, ntotal_param=False)

tt.plotting.plot_to_compare(df33_Cyclon_CPM_profile, df34_RPP25L_CPM_profile, figsize=(10,3),
                            title=f"Cyclon vs RPP25L reads along rDNA (RNA5S9)",
                            ref_label='Cyclon',label='RPP25L',
                            dpi=300,ylim=(None,1500),offset=100,fname=f'Fig_03_RNA5S9_Cyclon_vs_RPP25L.png')